# Validation notebook: comparison of rock energy segmentation methods

Вход: `united_rock_energy_features_all_methods.csv`

Notebook сравнивает пять способов разметки энергоёмкости:

1. `energy_type_kmeans`
2. `energy_type_gmm`
3. `energy_type_quantile`
4. `energy_type_rule_based`
5. `energy_type_segment_quantile`

Валидация:
- физическая осмысленность классов;
- control leakage check;
- temporal stability;
- переносимость между wells;
- полезность labels/hardness для прогноза ROP.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

RANDOM_STATE = 42
EPS = 1e-6

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Загрузка датасета со всеми методами разметки

In [ ]:
DATA_PATH = "united_rock_energy_features_all_methods.csv"

df = pd.read_csv(DATA_PATH)
df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

segmentation_methods = [
    "energy_type_kmeans",
    "energy_type_gmm",
    "energy_type_quantile",
    "energy_type_rule_based",
    "energy_type_segment_quantile",
]

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "formation_residual",
    "hardness_score_smooth",
] + segmentation_methods

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Не хватает колонок: {missing}")

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df[required_cols].head())

## 2. Универсальные функции валидации

In [ ]:
def summarize_energy_clusters(data, type_col):
    return (
        data
        .groupby(type_col)
        .agg(
            rows=("speed", "size"),
            speed_median=("speed", "median"),
            speed_mean=("speed", "mean"),
            pseudo_mse_median=("pseudo_mse", "median"),
            pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
            efficiency_median=("drilling_efficiency", "median"),
            efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
            residual_median=("formation_residual", "median"),
            residual_smooth=("formation_residual_roll_median_60", "median"),
            relative_residual_smooth=("relative_formation_residual_roll_median_60", "median"),
            hardness_median=("hardness_score", "median"),
            hardness_smooth=("hardness_score_smooth", "median"),
            speed_std_smooth=("speed_roll_std_60", "median"),
            rotation_std_smooth=("rotation_roll_std_60", "median"),
            pressure_axis_median=("pressure_axis", "median"),
            pressure_rotation_median=("pressure_rotation", "median"),
            rotation_median=("rotation", "median"),
            energy_input_median=("energy_input_proxy", "median"),
        )
        .sort_values("hardness_smooth")
    )


def physical_score(summary_df):
    # Чем лучше монотонность soft->hard, тем больше score.
    # Ожидаем: pseudo_mse ↑, hardness ↑, speed ↓, efficiency ↓, residual ↓.
    s = summary_df.copy().sort_values("hardness_smooth")
    score = 0

    def mono_inc(x):
        return np.all(np.diff(x) >= 0)

    def mono_dec(x):
        return np.all(np.diff(x) <= 0)

    score += int(mono_inc(s["hardness_smooth"].values))
    score += int(mono_inc(s["pseudo_mse_smooth"].values))
    score += int(mono_dec(s["efficiency_smooth"].values))
    score += int(mono_dec(s["residual_smooth"].values))
    score += int(mono_dec(s["speed_median"].values))

    return score


def control_leakage_check(data, type_col):
    return (
        data
        .groupby(type_col)
        .agg(
            rows=("speed", "size"),
            pressure_axis_median=("pressure_axis", "median"),
            pressure_axis_q25=("pressure_axis", lambda s: s.quantile(0.25)),
            pressure_axis_q75=("pressure_axis", lambda s: s.quantile(0.75)),
            pressure_rotation_median=("pressure_rotation", "median"),
            pressure_rotation_q25=("pressure_rotation", lambda s: s.quantile(0.25)),
            pressure_rotation_q75=("pressure_rotation", lambda s: s.quantile(0.75)),
            rotation_median=("rotation", "median"),
            speed_median=("speed", "median"),
            pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
            efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
            residual_smooth=("formation_residual_roll_median_60", "median"),
            hardness_smooth=("hardness_score_smooth", "median"),
        )
        .sort_values("hardness_smooth")
    )


def temporal_stability(data, type_col):
    tmp = data.sort_values(["well_id", "processing_time"]).copy()
    tmp["prev_type"] = tmp.groupby("well_id")[type_col].shift(1)
    tmp["changed"] = (tmp[type_col] != tmp["prev_type"]) & tmp["prev_type"].notna()

    report = (
        tmp.groupby("well_id")
           .agg(
               rows=(type_col, "size"),
               changes=("changed", "sum"),
           )
    )

    report["changes_per_1000_points"] = 1000 * report["changes"] / report["rows"]
    return report


def per_well_class_consistency(data, type_col):
    report = (
        data
        .groupby(["well_id", type_col])
        .agg(
            rows=("speed", "size"),
            pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
            efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
            residual_smooth=("formation_residual_roll_median_60", "median"),
            hardness_smooth=("hardness_score_smooth", "median"),
        )
        .reset_index()
    )

    consistency = (
        report
        .groupby(type_col)
        .agg(
            wells=("well_id", "nunique"),
            median_hardness=("hardness_smooth", "median"),
            std_hardness=("hardness_smooth", "std"),
            median_pseudo_mse=("pseudo_mse_smooth", "median"),
            std_pseudo_mse=("pseudo_mse_smooth", "std"),
            median_residual=("residual_smooth", "median"),
            std_residual=("residual_smooth", "std"),
        )
        .sort_values("median_hardness")
    )

    return consistency

## 3. Сводки и физическая монотонность по всем методам

In [ ]:
summaries = {}
physical_scores = []

for method in segmentation_methods:
    print("\n" + "=" * 100)
    print(method)
    summary = summarize_energy_clusters(df, method)
    summaries[method] = summary
    display(summary)

    score = physical_score(summary)
    physical_scores.append({"method": method, "physical_monotonicity_score_0_5": score})

physical_scores_df = pd.DataFrame(physical_scores).sort_values("physical_monotonicity_score_0_5", ascending=False)
display(physical_scores_df)

## 4. Control leakage check

In [ ]:
for method in segmentation_methods:
    print("\n" + "=" * 100)
    print(method)
    display(control_leakage_check(df, method))

## 5. Temporal stability comparison

In [ ]:
stability_rows = []
stability_reports = {}

for method in segmentation_methods:
    report = temporal_stability(df, method)
    stability_reports[method] = report

    desc = report["changes_per_1000_points"].describe(percentiles=[.05, .5, .95])

    stability_rows.append({
        "method": method,
        "mean_changes_per_1000": desc["mean"],
        "median_changes_per_1000": desc["50%"],
        "p95_changes_per_1000": desc["95%"],
        "max_changes_per_1000": desc["max"],
    })

stability_compare = pd.DataFrame(stability_rows).sort_values("median_changes_per_1000")
display(stability_compare)

for method in segmentation_methods:
    print("\n" + "=" * 100)
    print(method)
    display(stability_reports[method].describe(percentiles=[.05, .5, .95]))

## 6. Переносимость между wells

In [ ]:
for method in segmentation_methods:
    print("\n" + "=" * 100)
    print(method)
    display(per_well_class_consistency(df, method))

## 7. Визуализация по одному well_id

In [ ]:
sample_well = df["well_id"].dropna().unique()[0]
tmp = df[df["well_id"] == sample_well].copy()

for type_col in segmentation_methods:
    code_map = {v: i for i, v in enumerate(sorted(tmp[type_col].dropna().unique()))}
    colors = tmp[type_col].map(code_map)

    plt.figure(figsize=(14, 5))
    plt.scatter(tmp["processing_time"], tmp["speed"], c=colors, s=8, alpha=0.8)
    plt.xlabel("time")
    plt.ylabel("speed")
    plt.title(f"Speed colored by {type_col} | well_id={sample_well}")
    plt.grid(True, alpha=0.3)
    plt.show()

plt.figure(figsize=(14, 5))
plt.plot(tmp["processing_time"], tmp["hardness_score_smooth"])
plt.xlabel("time")
plt.ylabel("hardness_score_smooth")
plt.title(f"Hardness score over time | well_id={sample_well}")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Тест полезности labels/hardness для прогноза ROP

In [ ]:
prediction_df = df.dropna(subset=[
    "speed",
    "hardness_score_smooth",
] + segmentation_methods).copy()

base_features = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "total_pressure",
    "pressure_balance",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "rotation_efficiency",
    "axis_x_rotation",
    "rot_pressure_x_rotation",
    "energy_input_proxy",
    "log_energy_input_proxy",
]

hardness_features = base_features + [
    "hardness_score_smooth",
    "pseudo_mse_roll_median_60",
    "drilling_efficiency_roll_median_60",
    "formation_residual_roll_median_60",
]

target = "speed"

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(prediction_df, groups=prediction_df["well_id"]))

train_pred = prediction_df.iloc[train_idx].copy()
test_pred = prediction_df.iloc[test_idx].copy()

def fit_eval_numeric(features, model_name):
    model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        ))
    ])

    model.fit(train_pred[features], train_pred[target])
    pred = model.predict(test_pred[features])

    return {
        "model": model_name,
        "MAE": mean_absolute_error(test_pred[target], pred),
        "RMSE": root_mean_squared_error(test_pred[target], pred),
        "R2": r2_score(test_pred[target], pred),
    }

results = []
results.append(fit_eval_numeric(base_features, "controls_only"))
results.append(fit_eval_numeric(hardness_features, "controls_plus_hardness_score"))

# Добавляем categorical label каждого метода поверх base_features
for method in segmentation_methods:
    features_num = base_features
    categorical_features = [method]

    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), features_num),
        ],
        remainder="drop",
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        ))
    ])

    model.fit(train_pred[features_num + categorical_features], train_pred[target])
    pred = model.predict(test_pred[features_num + categorical_features])

    results.append({
        "model": f"controls_plus_{method}",
        "MAE": mean_absolute_error(test_pred[target], pred),
        "RMSE": root_mean_squared_error(test_pred[target], pred),
        "R2": r2_score(test_pred[target], pred),
    })

compare = pd.DataFrame(results).sort_values("MAE")
display(compare)

baseline = compare[compare["model"] == "controls_only"].iloc[0]

compare_improvement = compare.copy()
compare_improvement["MAE_improvement_vs_controls"] = baseline["MAE"] - compare_improvement["MAE"]
compare_improvement["RMSE_improvement_vs_controls"] = baseline["RMSE"] - compare_improvement["RMSE"]
compare_improvement["R2_improvement_vs_controls"] = compare_improvement["R2"] - baseline["R2"]

display(compare_improvement.sort_values("R2_improvement_vs_controls", ascending=False))

## 9. Итоговая таблица выбора метода

In [ ]:
method_selection_rows = []

for method in segmentation_methods:
    summary = summaries[method]
    stability = stability_compare[stability_compare["method"] == method].iloc[0]
    phys = physical_scores_df[physical_scores_df["method"] == method].iloc[0]

    pred_row = compare_improvement[compare_improvement["model"] == f"controls_plus_{method}"].iloc[0]

    method_selection_rows.append({
        "method": method,
        "physical_monotonicity_score_0_5": phys["physical_monotonicity_score_0_5"],
        "median_changes_per_1000": stability["median_changes_per_1000"],
        "p95_changes_per_1000": stability["p95_changes_per_1000"],
        "MAE_improvement_vs_controls": pred_row["MAE_improvement_vs_controls"],
        "R2_improvement_vs_controls": pred_row["R2_improvement_vs_controls"],
    })

method_selection = pd.DataFrame(method_selection_rows)

display(
    method_selection.sort_values(
        ["physical_monotonicity_score_0_5", "median_changes_per_1000", "R2_improvement_vs_controls"],
        ascending=[False, True, False],
    )
)

## 10. Сохранение validation report

In [ ]:
VALIDATION_REPORT_PATH = "rock_energy_all_methods_validation_report.json"

report = {
    "data_path": DATA_PATH,
    "segmentation_methods": segmentation_methods,
    "physical_scores": physical_scores_df.to_dict(orient="records"),
    "stability_compare": stability_compare.to_dict(orient="records"),
    "prediction_comparison": compare.to_dict(orient="records"),
    "prediction_improvement": compare_improvement.to_dict(orient="records"),
    "method_selection": method_selection.to_dict(orient="records"),
}

with open(VALIDATION_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Saved:", VALIDATION_REPORT_PATH)

## 11. Как выбирать финальный метод

Смотреть нужно не на одну метрику, а на компромисс:

- высокая физическая монотонность;
- низкое `median_changes_per_1000`;
- hard-класс не должен быть просто максимальным pressure;
- хорошая переносимость между wells;
- `controls_plus_<method>` должен улучшать прогноз относительно `controls_only`.

Ожидаемый кандидат: `energy_type_segment_quantile`, если он даст лучшую temporal stability и физическую монотонность.